In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp

# read xlsx
path = 'problem_2/problem_2.xlsx'
try:
    df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)
except FileNotFoundError:
    path = path.split('/')[1]
df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)

ModuleNotFoundError: No module named 'pandas'

In [2]:
df

,0,1,2,3,4,5,6,7
0,Courses,Taken?,OR req,Math req,Computer req,Prerequisites needed,Prerequisites taken,Prerequisites satisfied
1,Calculus,1,NaN,1,NaN,0,0,1
2,Operations Research,1,1,1,NaN,0,0,1
3,Data Structures,0,NaN,1,1,1,1,1
4,Business Statistics,0,1,1,NaN,1,1,1
5,Computer Simulation,1,1,NaN,1,1,1,1
6,Introduction to Computer Programming,1,NaN,NaN,1,0,0,1
7,Forecasting,0,1,1,NaN,1,0,0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Prerequisites,Calculus,Operations Research,Data Structures,Business Statistics,Computer Simulation,Introduction to Computer Programming,Forecasting


In [3]:
courses = df.iloc[1:8, 0]
list(courses)

['Calculus',
 'Operations Research',
 'Data Structures',
 'Business Statistics',
 'Computer Simulation',
 'Introduction to Computer Programming',
 'Forecasting']

In [4]:
requirements_fulfilled = df.iloc[1:8, 2:5].fillna(0)
requirements_fulfilled

,2,3,4
1,0,1,0
2,1,1,0
3,0,1,1
4,1,1,0
5,1,0,1
6,0,0,1
7,1,1,0


In [5]:
prerequisites = df.iloc[10:17, 1:8].fillna(0)
prerequisites.columns = df.iloc[10:17, 0]
prerequisites.index = df.iloc[10:17, 0]
prerequisites.index.name = 'Course'
prerequisites.columns.name = 'Prerequisite Course'
prerequisites

Prerequisite Course,Calculus,Operations Research,Data Structures,Business Statistics,Computer Simulation,Introduction to Computer Programming,Forecasting
Course,,,,,,,
Calculus,0,0,0,0,0,0,0
Operations Research,0,0,0,0,0,0,0
Data Structures,0,0,0,0,0,1,0
Business Statistics,1,0,0,0,0,0,0
Computer Simulation,0,0,0,0,0,1,0
Introduction to Computer Programming,0,0,0,0,0,0,0
Forecasting,0,0,0,1,0,0,0


In [6]:
minimun_courses_for_field = df.iloc[19:22, 1]
minimun_courses_for_field

19    2
20    2
21    2
Name: 1, dtype: object

In [7]:
model = gp.Model("p1")
x = {}
for course in courses:
    x[course] = model.addVar(vtype='B', name=course)
model.update()
model.setObjective(gp.quicksum(x[course] for course in courses), gp.GRB.MINIMIZE)
model.getVars()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-11-27


[<gurobi.Var Calculus>,
 <gurobi.Var Operations Research>,
 <gurobi.Var Data Structures>,
 <gurobi.Var Business Statistics>,
 <gurobi.Var Computer Simulation>,
 <gurobi.Var Introduction to Computer Programming>,
 <gurobi.Var Forecasting>]

In [8]:
requirements_actually_fulfilled = gp.LinExpr() + (requirements_fulfilled.values * np.array([x[course] for course in courses])[:, None]).sum(0)
for i, requirement in enumerate(requirements_actually_fulfilled):
    model.addConstr(requirement >= minimun_courses_for_field.values[i], name=f'requirements_actually_fulfilled_{i}')
# model.addConstr(requirements_actually_fulfilled >= minimun_courses_for_field.values, name='requirements_actually_fulfilled')

In [9]:
for course, prerequisite_courses in prerequisites.iterrows():
    for prerequisite_course, prerequisite in enumerate(prerequisite_courses):
        if prerequisite != 0:
            model.addConstr(x[course] <= x[courses.values[prerequisite_course]], name=f'{course} <= {prerequisite_course}')

In [10]:
model.update()
model.optimize()

print("\nThe optimal solutions:")
for var in model.getVars():
    print(f"{var.VarName}: {var.X}")
print(f"The optimal number of courses to take is:{model.objVal}")
# list_of_variables_defined_above = [total_sold, total_quality, expected_total_quality, rev_a, rev_b, rev_c, total_rev, prod_a, prod_b, used_a, used_b, unused_a, unused_b, total_cost, total_profit]
# list_of_variables_defined_above = [unused_a, unused_b]
# for x in list_of_variables_defined_above:
#     print(f"{x}: {x.getValue()}")
# for constr in model.getConstrs():
#     print(f"Constraint: {constr.ConstrName}, Dual Value: {constr.Pi}")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: 12th Gen Intel(R) Core(TM) i5-12500H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 7 rows, 7 columns and 20 nonzeros
Model fingerprint: 0x863e6fd0
Variable types: 0 continuous, 7 integer (7 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]
Found heuristic solution: objective 4.0000000
Presolve removed 2 rows and 1 columns
Presolve time: 0.01s
Presolved: 5 rows, 6 columns, 15 nonzeros
Variable types: 0 continuous, 6 integer (6 binary)

Root relaxation: cutoff, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0         4.00000    4.00000  0.00%

### write to excel

In [11]:
write_df = df.copy()
write_df.iloc[24, 1] = sum(list([int(val.X) for val in x.values()]))
write_df.iloc[1:8, 1] = list([int(val.X) for val in x.values()])
prerequisites_fullfilled = {}
for course, prerequisite_courses in prerequisites.iterrows():
    prerequisites_fullfilled[course] = 1
    for i, prerequisite_course in enumerate(prerequisite_courses):
        if prerequisite_course != 0 and courses.values[i] != course:
            if x[courses.values[i]].X == 0:
                prerequisites_fullfilled[course] = 0
                break
write_df.iloc[1:8, 7] = list(prerequisites_fullfilled.values())
write_df.iloc[19:22, 2] = [r.getValue() for r in requirements_actually_fulfilled]
write_df.iloc[0:8, 5] = write_df.iloc[0:8, 7]
write_df.iloc[0:8, 6:8] = np.nan

from openpyxl.styles import Font
from openpyxl import load_workbook

with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    write_df.to_excel(writer, sheet_name='Solution_Gurobi', index=False, header=False)
book = load_workbook(path)
sheet = book['Solution_Gurobi']


from openpyxl.utils import get_column_letter

source_sheet = book['Solution_Excel']
for i, column in enumerate(source_sheet.columns, start=1):
    letter = get_column_letter(i)
    width = source_sheet.column_dimensions[letter].width
    sheet.column_dimensions[letter].width = width


bold_font = Font(bold=True)

sheet['A25'].font = bold_font
sheet['B25'].font = bold_font

book.save(path)